In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/Consumer_Reviews_of_Amazon_Products.csv")
df = df.dropna(subset=['reviews.text', 'reviews.rating'])
df = df.drop_duplicates(subset=['reviews.text'])

def rating_to_sentiment(rating):
    if rating <= 2:
        return "Negative"
    elif rating == 3:
        return "Neutral"
    else:
        return "Positive"

df['sentiment'] = df['reviews.rating'].apply(rating_to_sentiment)

X = df['reviews.text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Keep negations — removing them flips the meaning of a review
NEGATIONS = {"not", "no", "nor", "never", "n't", "none", "nothing"}
STOP_WORDS = ENGLISH_STOP_WORDS - NEGATIONS

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)       # remove punctuation/digits
    text = re.sub(r"\s+", " ", text).strip()     # collapse whitespace
    words = [w for w in text.split() if w not in STOP_WORDS]
    return " ".join(words)

X_train_clean = X_train.apply(clean_text)
X_test_clean = X_test.apply(clean_text)

# Sanity check — compare before/after on one example
print("Original:", X_train.iloc[0])
print("Cleaned: ", X_train_clean.iloc[0])


from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X_train_tfidf = vectorizer.fit_transform(X_train_clean)
X_test_tfidf = vectorizer.transform(X_test_clean)

print(X_train_tfidf.shape, X_test_tfidf.shape)


from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
clf.fit(X_train_tfidf, y_train)


from sklearn.metrics import accuracy_score

y_pred = clf.predict(X_test_tfidf)

baseline_accuracy = accuracy_score(y_test, y_pred)
print(f"Baseline accuracy: {baseline_accuracy:.3f}")


sample = ["This product broke after two days, terrible quality"]
sample_clean = [clean_text(s) for s in sample]
sample_tfidf = vectorizer.transform(sample_clean)
print(clf.predict(sample_tfidf))

C:\Users\Admin\AppData\Local\Temp\ipykernel_5836\1975750478.py:4: DtypeWarning: Columns (1: name, 10: reviews.didPurchase) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/Consumer_Reviews_of_Amazon_Products.csv")


Original: We like the Fire better than Roku. We also prefer the ease of use over the funtionality of our new Sony 4K TV.
Cleaned:  like better roku prefer ease use funtionality new sony k tv
(27700, 5000) (6926, 5000)
Baseline accuracy: 0.841
['Negative']
